In [13]:
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Load data once (assume file exists in /data folder)
# ------------------------------------------------------------
df = pd.read_excel("data/combined.xlsx")
# Required columns – adjust if names differ
LOAD_COMM = "Community Load"
LOAD_RO   = "RO Load (kWh)"
PV_PER_KW = "Solar Power (kW)"
WT_PER_KW = "Wind Power (kW) (EO-20)"

# Total hourly load
load_total = df[LOAD_COMM] + df[LOAD_RO]

# Hourly generation per kW installed
pv_per_kw = df[PV_PER_KW].values
wt_per_kw = df[WT_PER_KW].values

# ------------------------------------------------------------
# Fixed system parameters (change as needed)
# ------------------------------------------------------------
# Efficiencies
ELEC_EFF = 0.85          # electrolyser efficiency (hydrogen out / electricity in)
FC_EFF   = 0.60          # fuel cell efficiency (electricity out / hydrogen in)
DG_EFF   = 0.35          # diesel generator efficiency (kWhₑ / litre fuel)

# Storage limits
HS_MIN_FRAC = 0.2        # minimum state of charge (fraction of capacity)
HS_MAX_FRAC = 1.0        # maximum SOC (fraction of capacity)

# Diesel minimum power fraction of rated capacity
DG_MIN_FRAC = 0.3

# Economic & emission parameters (annualised capital, O&M, fuel, grid)
PV_CAP_COST    = 1200    # $/kW
WT_CAP_COST    = 1500    # $/kW
HS_CAP_COST    = 500     # $/kWh (hydrogen storage)
FC_CAP_COST    = 800     # $/kW (fuel cell)
ELEC_CAP_COST  = 700     # $/kW (electrolyser)
DG_CAP_COST    = 400     # $/kW
GRID_CAP_COST  = 0       # $/kW (grid connection cost, if any)

PV_OM_FIX      = 10      # $/kW/yr
WT_OM_FIX      = 20      # $/kW/yr
HS_OM_FIX      = 5       # $/kWh/yr
FC_OM_FIX      = 15      # $/kW/yr
ELEC_OM_FIX    = 10      # $/kW/yr
DG_OM_FIX      = 25      # $/kW/yr
GRID_OM_FIX    = 0

DG_VAR_COST    = 0.4     # $/kWh (fuel + variable O&M)
GRID_PRICE     = 0.15    # $/kWh purchased

# Emission factors
DG_EMISSION    = 0.8     # kg CO₂/kWh generated
GRID_EMISSION  = 0.5     # kg CO₂/kWh imported

# Financial
INTEREST_RATE  = 0.08    # discount rate
LIFETIME       = 20      # years

# Annualisation factor (capital recovery factor)
CRF = INTEREST_RATE * (1 + INTEREST_RATE)**LIFETIME / ((1 + INTEREST_RATE)**LIFETIME - 1)

# ------------------------------------------------------------
# Objective function
# ------------------------------------------------------------
def objective(x):
    """
    x = [PV_kW, WT_kW, HS_kWh, FC_kW, Elec_kW, DG_kW, Grid_kW]
    Returns: [total_annual_cost ($), total_annual_emissions (kg CO₂)]
    """
    # Unpack decision variables
    PV_cap, WT_cap, HS_cap, FC_cap, Elec_cap, DG_cap, Grid_limit = x

    # Initial state of charge (start at 50% of capacity)
    SOC = HS_cap * 0.5
    HS_min = HS_cap * HS_MIN_FRAC
    HS_max = HS_cap * HS_MAX_FRAC

    # Accumulators
    total_load = 0.0
    total_unmet = 0.0
    dg_fuel_cost = 0.0
    dg_emissions = 0.0
    grid_cost = 0.0
    grid_emissions = 0.0
    dg_energy = 0.0      # for variable O&M if needed
    grid_energy = 0.0

    # Hourly simulation
    for t in range(len(load_total)):
        load = load_total.iloc[t]
        total_load += load

        # Renewable generation
        Epv = PV_cap * pv_per_kw[t]
        Ewt = WT_cap * wt_per_kw[t]
        E = Epv + Ewt
        deficit = load - E   # positive = shortage, negative = excess

        if deficit <= 0:
            # ---------- Excess: charge hydrogen ----------
            excess = -deficit
            charge_power = min(excess, Elec_cap)   # electrolyser limit
            # Energy added to storage (electrical → hydrogen)
            h2_added = charge_power * ELEC_EFF      # kWh (hydrogen energy)
            # Available space in storage
            space = HS_max - SOC
            actual_h2 = min(h2_added, space)
            SOC += actual_h2
            # Remaining excess is curtailed (no cost)

        else:
            # ---------- Deficit: supply from storage, DG, grid ----------
            unmet = deficit

            # 1. Fuel cell (hydrogen storage)
            if FC_cap > 0 and SOC > HS_min:
                # Maximum electrical energy from FC this hour
                max_fc_elec = FC_cap * 1.0
                # Available hydrogen energy (kWh) above minimum
                avail_h2 = SOC - HS_min
                # Corresponding electrical energy possible
                max_fc_from_h2 = avail_h2 * FC_EFF
                # Actual FC supply (electrical kWh)
                fc_supply = min(unmet, max_fc_elec, max_fc_from_h2)
                unmet -= fc_supply
                # Hydrogen consumed
                h2_used = fc_supply / FC_EFF
                SOC -= h2_used

            # 2. Diesel generator
            if unmet > 0 and DG_cap > 0:
                dg_min_power = DG_cap * DG_MIN_FRAC
                # Can we meet at least the minimum output?
                if unmet >= dg_min_power:
                    dg_supply = min(unmet, DG_cap)
                    unmet -= dg_supply
                    # Fuel consumption (litres) and cost / emissions
                    fuel_l = dg_supply / DG_EFF
                    dg_fuel_cost += fuel_l * DG_VAR_COST
                    dg_emissions += dg_supply * DG_EMISSION
                    dg_energy += dg_supply

            # 3. Grid import
            if unmet > 0 and Grid_limit > 0:
                grid_supply = min(unmet, Grid_limit)
                unmet -= grid_supply
                grid_cost += grid_supply * GRID_PRICE
                grid_emissions += grid_supply * GRID_EMISSION
                grid_energy += grid_supply

            # 4. Still unmet -> record
            if unmet > 0:
                total_unmet += unmet

    # ---------- Performance metrics ----------
    lpsp = total_unmet / total_load if total_load > 0 else 0.0

    # ---------- Annualised capital costs ----------
    cap_cost = (PV_cap * PV_CAP_COST +
                WT_cap * WT_CAP_COST +
                HS_cap * HS_CAP_COST +
                FC_cap * FC_CAP_COST +
                Elec_cap * ELEC_CAP_COST +
                DG_cap * DG_CAP_COST +
                Grid_limit * GRID_CAP_COST) * CRF

    # ---------- Fixed O&M ----------
    om_fixed = (PV_cap * PV_OM_FIX +
                WT_cap * WT_OM_FIX +
                HS_cap * HS_OM_FIX +
                FC_cap * FC_OM_FIX +
                Elec_cap * ELEC_OM_FIX +
                DG_cap * DG_OM_FIX +
                Grid_limit * GRID_OM_FIX)

    # ---------- Variable costs (already accumulated) ----------
    # (dg_fuel_cost includes variable O&M)
    var_cost = dg_fuel_cost + grid_cost

    # ---------- Total annual cost ----------
    total_cost = cap_cost + om_fixed + var_cost

    # ---------- Total annual emissions ----------
    total_emissions = dg_emissions + grid_emissions

    # For multi‑objective optimisation, return the two objectives.
    # LPSP is treated as a constraint outside this function.
    return [total_cost, total_emissions]

In [14]:
# ============================================================
# Quick test with LPSP calculation
# ============================================================

def quick_check(x):
    # Unpack
    PV_cap, WT_cap, HS_cap, FC_cap, Elec_cap, DG_cap, Grid_limit = x
    
    # Initial state
    SOC = HS_cap * 0.5
    HS_min = HS_cap * HS_MIN_FRAC
    HS_max = HS_cap * HS_MAX_FRAC
    
    # Trackers
    total_load = 0
    total_unmet = 0
    unmet_hours = 0
    
    # Hourly simulation
    for t in range(len(load_total)):
        load = load_total.iloc[t]
        total_load += load
        
        # Renewable generation
        Epv = PV_cap * pv_per_kw[t]
        Ewt = WT_cap * wt_per_kw[t]
        E = Epv + Ewt
        deficit = load - E
        
        if deficit <= 0:
            # Excess - charge hydrogen
            excess = -deficit
            charge_power = min(excess, Elec_cap)
            h2_added = charge_power * ELEC_EFF
            space = HS_max - SOC
            actual_h2 = min(h2_added, space)
            SOC += actual_h2
        else:
            # Deficit - try to meet load
            unmet = deficit
            
            # 1. Fuel cell
            if FC_cap > 0 and SOC > HS_min:
                max_fc_elec = FC_cap
                avail_h2 = SOC - HS_min
                max_fc_from_h2 = avail_h2 * FC_EFF
                fc_supply = min(unmet, max_fc_elec, max_fc_from_h2)
                unmet -= fc_supply
                SOC -= fc_supply / FC_EFF
            
            # 2. Diesel
            if unmet > 0 and DG_cap > 0:
                dg_min_power = DG_cap * DG_MIN_FRAC
                if unmet >= dg_min_power:
                    dg_supply = min(unmet, DG_cap)
                    unmet -= dg_supply
            
            # 3. Grid
            if unmet > 0 and Grid_limit > 0:
                grid_supply = min(unmet, Grid_limit)
                unmet -= grid_supply
            
            # 4. Still unmet -> record
            if unmet > 0:
                total_unmet += unmet
                unmet_hours += 1
    
    # Calculate LPSP
    lpsp = total_unmet / total_load if total_load > 0 else 0
    print(f"Total annual load: {total_load:,.0f} kWh")
    print(f"Total unmet load:  {total_unmet:,.0f} kWh")
    print(f"Hours with unmet load: {unmet_hours} out of 8760")
    print(f"LPSP: {lpsp:.4f} ({lpsp*100:.2f}%)")
    
    if lpsp == 0:
        print("✅ Load was fully met all year")
    elif lpsp < 0.01:
        print(f"⚠️  Load met 99% of the time (LPSP = {lpsp:.4f})")
    else:
        print(f"❌ Significant unmet load (LPSP = {lpsp:.4f})")
    
    return lpsp, total_unmet, unmet_hours

# 

In [15]:
# ============================================================
# Quick test for the hybrid system objective function
# ============================================================

# Candidate decision variables: 
# [PV_kW, WT_kW, HS_kWh, FC_kW, Elec_kW, DG_kW, Grid_kW]
x_test = [100,    # 100 kW solar
          50,     # 50 kW wind
          200,    # 200 kWh hydrogen storage
          30,     # 30 kW fuel cell
          40,     # 40 kW electrolyser
          20,     # 20 kW diesel generator
          50000]     # 50 kW grid import limit

# Run the objective function
cost, emissions = objective(x_test)

# Print results
print("===== Quick Check =====")
print(f"PV: {x_test[0]:.0f} kW")
print(f"WT: {x_test[1]:.0f} kW")
print(f"HS: {x_test[2]:.0f} kWh")
print(f"FC: {x_test[3]:.0f} kW")
print(f"Elec: {x_test[4]:.0f} kW")
print(f"DG: {x_test[5]:.0f} kW")
print(f"Grid: {x_test[6]:.0f} kW")
print("-----------------------")
print(f"Total annual cost: ${cost:,.2f}")
print(f"Total annual emissions: {emissions:,.0f} kg CO₂")


print("\n--- Load met analysis ---")
lpsp, unmet, hours = quick_check(x_test)

===== Quick Check =====
PV: 100 kW
WT: 50 kW
HS: 200 kWh
FC: 30 kW
Elec: 40 kW
DG: 20 kW
Grid: 50000 kW
-----------------------
Total annual cost: $407,944.46
Total annual emissions: 866,881 kg CO₂

--- Load met analysis ---
Total annual load: 3,134,336 kWh
Total unmet load:  0 kWh
Hours with unmet load: 0 out of 8760
LPSP: 0.0000 (0.00%)
✅ Load was fully met all year
